# TorGen: DETR-CVAE for Tornado Outbreak Generation

Install the package, mount Drive, and train.

In [ ]:
!pip install -q --no-cache-dir git+https://github.com/jcaramichaellehigh/TorGen.git

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from torgen.training import TrainConfig, train

config = TrainConfig(
    drive_dir="/content/drive/MyDrive/raw",
    checkpoint_dir="/content/drive/MyDrive/checkpoints",
    ef_weight_power=0.5,  # 0 = uniform, 0.5 = sqrt IF, 1 = full inverse-frequency
)

trainer = train(config)

In [ ]:
print(f"Epochs trained: {trainer.epoch}")
print(f"Final train loss: {trainer.train_losses[-1]:.4f}")
print(f"Final val loss: {trainer.val_losses[-1]:.4f}")
print(f"Best val loss: {trainer.best_val_loss:.4f}")
if trainer.loss_history["train_kl"]:
    print(f"Final train recon: {trainer.loss_history['train_recon'][-1]:.4f}")
    print(f"Final train KL: {trainer.loss_history['train_kl'][-1]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

h = trainer.loss_history
has_decomposed = len(h["train_kl"]) > 0

if has_decomposed:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    epochs = range(1, len(h["train_total"]) + 1)

    # Total loss
    axes[0].plot(epochs, h["train_total"], label="Train")
    axes[0].plot(epochs, h["val_total"], label="Val")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Total Loss (recon + β·KL)")
    axes[0].legend()

    # Recon loss
    axes[1].plot(epochs, h["train_recon"], label="Train")
    axes[1].plot(epochs, h["val_recon"], label="Val")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Reconstruction Loss")
    axes[1].legend()

    # KL divergence
    axes[2].plot(epochs, h["train_kl"], label="Train KL")
    axes[2].plot(epochs, h["val_kl"], label="Val KL")
    ax2 = axes[2].twinx()
    ax2.plot(epochs, h["beta"], "k--", alpha=0.3, label="β")
    ax2.set_ylabel("β")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("KL Divergence")
    axes[2].set_title("KL Divergence + β Schedule")
    axes[2].legend(loc="upper left")
    ax2.legend(loc="upper right")

    fig.tight_layout()
    plt.show()
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, len(trainer.train_losses) + 1), trainer.train_losses, label="Train")
    ax.plot(range(1, len(trainer.val_losses) + 1), trainer.val_losses, label="Val")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title("TorGen Training Curves")
    ax.legend()
    fig.tight_layout()
    plt.show()

In [ ]:
import json
import os

eval_dir = os.path.join(config.checkpoint_dir, "eval")
summary_path = os.path.join(eval_dir, "summary.json")
if os.path.exists(summary_path):
    with open(summary_path) as f:
        results = json.load(f)
    print("Evaluation Results:")
    for k, v in results.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
else:
    print("No evaluation results found (evaluation runs automatically after training)")